# **Installs**

In [ ]:
!pip -q install pdfplumber pandas nltk tqdm

# **Imports**

In [ ]:
import re
import json
import random
from pathlib import Path

import pdfplumber
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ====== CONFIG ======
RAW_PDF_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Transcripts")   # your uploaded PDFs are here
PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")

RAW_TEXT_DIR = PROJECT_DIR / "raw_text"
CLEAN_TXT_DIR = PROJECT_DIR / "cleaned_transcripts"
SENTENCE_DIR = PROJECT_DIR / "sentence_data"
CHUNK_DIR = PROJECT_DIR / "chunk_data"
RETRIEVER_DIR = PROJECT_DIR / "retriever_data"
LLM_DIR = PROJECT_DIR / "llm_data"

for d in [RAW_TEXT_DIR, CLEAN_TXT_DIR, SENTENCE_DIR, CHUNK_DIR, RETRIEVER_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Folders ready:", PROJECT_DIR)

Folders ready: /content/drive/MyDrive/DSA4265_Project/Results


In [ ]:
pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))
for f in pdf_files:
    print(f.name)

# **Chunking for RAG v2**

In [ ]:
import pandas as pd
import json
from pathlib import Path
from nltk.tokenize import sent_tokenize

# --- Paths ---
PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")
CHUNK_DIR = PROJECT_DIR / "chunk_data"
turns_csv_path = PROJECT_DIR / "all_speaker_turns.csv"

# --- Load existing speaker turns ---
turns_df = pd.read_csv(turns_csv_path)
print("Loaded turns_df with", len(turns_df), "rows")

# --- v2 chunking function ---
def create_chunks_from_turns_v2(df, turns_per_chunk=3, overlap=1, max_chars=1500):
    chunk_rows = []

    for company_quarter, group in df.groupby("company_quarter"):
        group = group.sort_values("turn_id").reset_index(drop=True)

        company = group.loc[0, "company"]
        quarter = group.loc[0, "quarter"]
        year    = group.loc[0, "year"]

        # Step 1: expand long turns into sentence-level sub-turns
        expanded_turns = []
        for _, row in group.iterrows():
            if len(row["text"]) <= max_chars:
                expanded_turns.append(row.to_dict())
            else:
                sentences = sent_tokenize(row["text"])
                buffer, buf_len = [], 0
                sub_id = 0
                for sent in sentences:
                    if buf_len + len(sent) > max_chars and buffer:
                        new_row = row.to_dict()
                        new_row["text"] = " ".join(buffer)
                        new_row["turn_id"] = row["turn_id"] + sub_id * 0.01
                        expanded_turns.append(new_row)
                        buffer, buf_len = [], 0
                        sub_id += 1
                    buffer.append(sent)
                    buf_len += len(sent)
                if buffer:
                    new_row = row.to_dict()
                    new_row["text"] = " ".join(buffer)
                    new_row["turn_id"] = row["turn_id"] + sub_id * 0.01
                    expanded_turns.append(new_row)

        # Step 2: sliding window chunking
        step = max(1, turns_per_chunk - overlap)
        chunk_id = 0

        for start in range(0, len(expanded_turns), step):
            end = start + turns_per_chunk
            window = expanded_turns[start:end]

            if not window:
                continue

            chunk_text_parts = []
            speakers = []
            for row in window:
                chunk_text_parts.append(f"[Speaker: {row['speaker']}] {row['text']}")
                speakers.append(row["speaker"])

            chunk_text = "\n".join(chunk_text_parts).strip()

            chunk_rows.append({
                "company":           company,
                "quarter":           quarter,
                "year":              year,
                "company_quarter":   company_quarter,
                "chunk_id":          f"{company_quarter}_chunk_{chunk_id}_v2",
                "start_turn_id":     window[0]["turn_id"],
                "end_turn_id":       window[-1]["turn_id"],
                "speakers_in_chunk": list(dict.fromkeys(speakers)),
                "chunk_text":        chunk_text
            })

            chunk_id += 1
            if end >= len(expanded_turns):
                break

    return pd.DataFrame(chunk_rows)

# --- Create and save v2 chunks ---
chunks_df_v2 = create_chunks_from_turns_v2(
    turns_df,
    turns_per_chunk=3,
    overlap=1,
    max_chars=1500
)

# Save CSV
chunks_df_v2.to_csv(CHUNK_DIR / "rag_chunks_v2.csv", index=False)

# Save JSONL
chunk_jsonl_path_v2 = CHUNK_DIR / "rag_chunks_v2.jsonl"
with open(chunk_jsonl_path_v2, "w", encoding="utf-8") as f:
    for _, row in chunks_df_v2.iterrows():
        record = row.to_dict()
        record["speakers_in_chunk"] = list(record["speakers_in_chunk"])
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Chunks v2 saved:", chunk_jsonl_path_v2)
print("Total chunks v2:", len(chunks_df_v2))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DSA4265_Project/Results/all_speaker_turns.csv'

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")
CHUNK_DIR = PROJECT_DIR / "chunk_data"

# load old + new
old_chunks = pd.read_csv(CHUNK_DIR / "rag_chunks.csv")
new_chunks = pd.read_csv(CHUNK_DIR / "rag_chunks_v2.csv")

print("Old chunk count:", len(old_chunks))
print("New chunk count:", len(new_chunks))
print()

# compare length distributions
old_chunks["chunk_length"] = old_chunks["chunk_text"].str.len()
new_chunks["chunk_length"] = new_chunks["chunk_text"].str.len()

print("=== OLD chunk lengths ===")
print(old_chunks["chunk_length"].describe())
print()

print("=== NEW v2 chunk lengths ===")
print(new_chunks["chunk_length"].describe())
print()

# specifically inspect chunk_0 issue
print("=== chunk_0 comparison ===")
old_chunk0 = old_chunks[old_chunks["chunk_id"].str.contains("chunk_0")]
new_chunk0 = new_chunks[new_chunks["chunk_id"].str.contains("chunk_0")]

print("Old chunk_0 avg length:", old_chunk0["chunk_length"].mean())
print("New chunk_0 avg length:", new_chunk0["chunk_length"].mean())

# **rag**

In [ ]:
from pathlib import Path

# ====== CONFIG ======
RAW_PDF_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Transcripts")   # your uploaded PDFs are here
PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")

RAW_TEXT_DIR = PROJECT_DIR / "raw_text"
CLEAN_TXT_DIR = PROJECT_DIR / "cleaned_transcripts"
SENTENCE_DIR = PROJECT_DIR / "sentence_data"
CHUNK_DIR = PROJECT_DIR / "chunk_data"
RETRIEVER_DIR = PROJECT_DIR / "retriever_data"
LLM_DIR = PROJECT_DIR / "llm_data"

for d in [RAW_TEXT_DIR, CLEAN_TXT_DIR, SENTENCE_DIR, CHUNK_DIR, RETRIEVER_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Folders ready:", PROJECT_DIR)

In [ ]:
!pip install faiss-cpu
!pip install sentence-transformers

In [ ]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Load your chunks
chunks_df = pd.read_csv(CHUNK_DIR / "rag_chunks_v2.csv")

# 2. Load embedding model
model = SentenceTransformer("all-mpnet-base-v2")

# 3. Embed all chunk texts
texts = chunks_df["chunk_text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# 4. Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(embeddings)
index.add(embeddings)

# 5. Save index + embeddings + metadata
faiss.write_index(index, str(RETRIEVER_DIR / "faiss.index"))
np.save(str(RETRIEVER_DIR / "embeddings.npy"), embeddings)   # <-- add this
chunks_df.to_csv(RETRIEVER_DIR / "chunks_with_index.csv", index=False)

print(f"Indexed {len(texts)} chunks")

In [ ]:
# --- Config (single source of truth) ---
MODEL_NAME     = "all-mpnet-base-v2"
TOP_K          = 5
OVERFETCH      = 250

# --- Load once at the top of your eval notebook ---
model      = SentenceTransformer(MODEL_NAME)
index      = faiss.read_index(str(RETRIEVER_DIR / "faiss.index"))
embeddings = np.load(str(RETRIEVER_DIR / "embeddings.npy"))       # <-- load saved array
chunks_df  = pd.read_csv(RETRIEVER_DIR / "chunks_with_index.csv")

# Guard: drop any rows with missing chunk text
chunks_df = chunks_df.dropna(subset=["chunk_text"]).reset_index(drop=True)

def retrieve(query, model, index, chunks_df, embeddings,
             top_k=TOP_K, overfetch=OVERFETCH,
             filter_company=None, filter_quarter=None, rerank=True):

    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    fetch_k = top_k * overfetch
    scores, indices = index.search(q_emb, fetch_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[idx]
        if filter_company and row["company"].lower() != filter_company.lower():
            continue
        if filter_quarter and row["quarter"] != filter_quarter:
            continue
        candidates.append({
            "chunk_id":   row["chunk_id"],
            "company":    row["company"],
            "quarter":    row["quarter"],
            "score":      float(score),
            "chunk_text": row["chunk_text"],
            "embedding":  embeddings[idx],
        })

    if rerank and candidates:
        candidate_embs = np.array([c["embedding"] for c in candidates])
        cos_sims = (candidate_embs @ q_emb.T).squeeze()
        if cos_sims.ndim == 0:
            cos_sims = np.array([cos_sims])
        for i, c in enumerate(candidates):
            c["score"] = float(cos_sims[i])
        candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    if len(candidates) < top_k:
        print(f"Warning: only {len(candidates)} candidates after filtering. "
              f"company={filter_company}, quarter={filter_quarter}")

    return candidates[:top_k]

# evaluation

In [ ]:
from pathlib import Path
print((RETRIEVER_DIR / "faiss.index").exists())

In [ ]:
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content/drive/MyDrive/DSA4265_Project")

CHUNK_PATH     = BASE_DIR / "Results" / "chunk_data" / "rag_chunks_v2.csv"
INDEX_PATH     = BASE_DIR / "Results" / "retriever_data" / "faiss.index"
LABELS_PATH    = BASE_DIR / "Results" / "retriever_data" / "starter_query_pairs_template_labelled_v2.csv"
EMBEDDINGS_PATH = BASE_DIR / "Results" / "retriever_data" / "embeddings.npy"

model = SentenceTransformer("all-mpnet-base-v2")
index = faiss.read_index(str(INDEX_PATH))
embeddings = np.load(str(EMBEDDINGS_PATH))
chunks = pd.read_csv(CHUNK_PATH)
labels = pd.read_csv(LABELS_PATH)

In [ ]:
def evaluate_retriever(labels_df, chunks_df, model, index, embeddings, top_k=5):
    precision_scores   = []
    recall_scores      = []
    reciprocal_ranks   = []
    hard_negative_wins = []
    per_query_results  = []

    # Group by query so each query is evaluated once with all its positives
    for (company, quarter, query), group in labels_df.groupby(["company", "quarter", "query"]):
        positive_ids     = set(group["positive_chunk_ids"].iloc[0]
                               if not isinstance(group["positive_chunk_ids"].iloc[0], str)
                               else ast.literal_eval(group["positive_chunk_ids"].iloc[0]))
        hard_negative_id = group["negative_chunk_id"].iloc[0]

        retrieved     = retrieve(query, model, index, chunks_df, embeddings,
                                 top_k=top_k, filter_company=company, filter_quarter=quarter)
        retrieved_ids = [r["chunk_id"] for r in retrieved]

        hits = [pid for pid in positive_ids if pid in retrieved_ids]
        hit  = 1 if hits else 0

        precision_scores.append(len(hits) / top_k)
        recall_scores.append(float(hit))

        if hits:
            rank = min(retrieved_ids.index(pid) for pid in hits if pid in retrieved_ids) + 1
            rr   = 1 / rank
        else:
            rank = None
            rr   = 0.0
        reciprocal_ranks.append(rr)

        if hit and hard_negative_id not in retrieved_ids:
            ranked_correctly = 1
        elif hit and hard_negative_id in retrieved_ids:
            pos_rank  = min(retrieved_ids.index(pid) for pid in hits)
            hard_rank = retrieved_ids.index(hard_negative_id)
            ranked_correctly = 1 if pos_rank < hard_rank else 0
        else:
            ranked_correctly = 0
        hard_negative_wins.append(ranked_correctly)

        per_query_results.append({
            "company":                    company,
            "quarter":                    quarter,
            "query":                      query,
            "positive_ids":               list(positive_ids),
            "hard_negative_id":           hard_negative_id,
            "retrieved_ids":              retrieved_ids,
            "hit":                        hit,
            "rank":                       rank,
            "reciprocal_rank":            round(rr, 4),
            "ranked_above_hard_negative": ranked_correctly,
        })

    metrics = {
        f"Precision@{top_k}": round(np.mean(precision_scores), 4),
        f"Recall@{top_k}":    round(np.mean(recall_scores), 4),
        "MRR":                round(np.mean(reciprocal_ranks), 4),
        "Hard neg acc":       round(np.mean(hard_negative_wins), 4),
    }

    return metrics, pd.DataFrame(per_query_results)

In [ ]:
labels = pd.read_csv(RETRIEVER_DIR / "starter_query_pairs_template_labelled_v2.csv")

k_values    = [5, 6, 7, 8, 9, 10]
all_metrics = []
for k in k_values:
    metrics, _ = evaluate_retriever(labels, chunks, model, index, embeddings, top_k=k)
    metrics["k"] = k
    all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics).set_index("k")
print(metrics_df.to_string())

updated code